# MNIST 차원축소 비교: PCA vs LDA vs MDS (3D Interactive)

세 가지 고전적 차원축소 기법으로 MNIST 숫자 이미지를 **3차원 공간**에 투영하고 Plotly로 인터랙티브하게 비교합니다.

| 기법 | 유형 | 특징 |
|------|------|------|
| **PCA** | 비지도 (선형) | 분산 최대 보존 방향 선택 |
| **LDA** | 지도 (선형) | 클래스 간 분리 최대화 |
| **MDS** | 비지도 (비선형) | 샘플 간 거리 구조 보존 |

## 0. 패키지 설치

In [ ]:
!pip install -q plotly scikit-learn

## 1. 임포트 및 하이퍼파라미터

In [ ]:
import time
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

# Colab 인라인 렌더링
pio.renderers.default = "colab"

# ── 하이퍼파라미터 ──────────────────────────
N_SAMPLES   = 3000   # 전체 사용 샘플 수
N_MDS       = 1500   # MDS 서브샘플 수 (O(n²) 비용 절감)
RANDOM_SEED = 42
OUTPUT_HTML = "mnist_dim_reduction_3d.html"

COLORS = [
    "#E74C3C",  # 0 — 빨강
    "#3498DB",  # 1 — 파랑
    "#2ECC71",  # 2 — 초록
    "#F39C12",  # 3 — 주황
    "#9B59B6",  # 4 — 보라
    "#1ABC9C",  # 5 — 청록
    "#E67E22",  # 6 — 진주황
    "#34495E",  # 7 — 짙은 회색
    "#E91E63",  # 8 — 핑크
    "#00BCD4",  # 9 — 하늘색
]

print(f"설정 완료 | N_SAMPLES={N_SAMPLES}, N_MDS={N_MDS}, SEED={RANDOM_SEED}")

## 2. 데이터 로드 및 전처리

클래스 불균형을 방지하기 위해 **클래스별 균등 서브샘플링**을 적용합니다.

In [ ]:
# ─── 데이터 로드 ───────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_all = np.concatenate([x_train, x_test], axis=0).reshape(-1, 784).astype("float32") / 255.0
y_all = np.concatenate([y_train, y_test], axis=0)

print(f"전체 샘플: {len(x_all):,}개  |  shape: {x_all.shape}")

# ─── 클래스별 균등 서브샘플링 ──────────────
rng = np.random.default_rng(RANDOM_SEED)
per_class = N_SAMPLES // 10
idx = []
for c in range(10):
    ci = np.where(y_all == c)[0]
    idx.append(rng.choice(ci, size=per_class, replace=False))
idx = np.concatenate(idx)
rng.shuffle(idx)

x, y = x_all[idx], y_all[idx]
print(f"서브샘플: {len(x):,}개  (클래스당 {per_class}개)")

# 클래스별 분포 확인
for d in range(10):
    print(f"  숫자 {d}: {(y == d).sum()}개")

## 3. PCA — 주성분 분석

분산이 가장 큰 방향(주성분)을 순서대로 선택해 3차원으로 투영합니다.  
**비지도 / 선형** 기법으로 레이블 정보를 사용하지 않습니다.

In [ ]:
# ─── PCA 3D 투영 ───────────────────────────
print("[PCA] 표준화 후 3D 투영 중...")
t0 = time.time()

scaler_pca = StandardScaler()
x_pca_scaled = scaler_pca.fit_transform(x)

pca = PCA(n_components=3, random_state=RANDOM_SEED)
pca_coords = pca.fit_transform(x_pca_scaled)

pca_elapsed = time.time() - t0
pca_var = pca.explained_variance_ratio_

print(f"  완료 ({pca_elapsed:.1f}s)")
print(f"  설명분산 — PC1: {pca_var[0]:.3f}, PC2: {pca_var[1]:.3f}, PC3: {pca_var[2]:.3f}")
print(f"  누적 설명분산: {pca_var.sum():.3f} ({pca_var.sum()*100:.1f}%)")

## 4. LDA — 선형 판별 분석

클래스 내 분산은 줄이고 클래스 간 분산은 최대화하는 방향을 찾습니다.  
**지도 / 선형** 기법으로 레이블을 활용하므로 클래스 분리가 선명합니다.  
> 최대 성분 수 = `n_classes - 1 = 9` → 3개 선택

In [ ]:
# ─── LDA 3D 투영 ───────────────────────────
print("[LDA] 표준화 후 3D 투영 중...")
t0 = time.time()

scaler_lda = StandardScaler()
x_lda_scaled = scaler_lda.fit_transform(x)

lda = LDA(n_components=3)
lda_coords = lda.fit_transform(x_lda_scaled, y)

lda_elapsed = time.time() - t0
lda_var = lda.explained_variance_ratio_

print(f"  완료 ({lda_elapsed:.1f}s)")
print(f"  설명분산 — LD1: {lda_var[0]:.3f}, LD2: {lda_var[1]:.3f}, LD3: {lda_var[2]:.3f}")
print(f"  누적 설명분산: {lda_var.sum():.3f} ({lda_var.sum()*100:.1f}%)")

## 5. MDS — 다차원 척도법

고차원 공간에서의 **샘플 간 거리 구조**를 3차원에서도 최대한 보존합니다.  
계산량이 O(n²)이므로 별도의 서브샘플(N_MDS)을 사용합니다.  
> **속도 개선 트릭:** PCA로 먼저 50차원으로 압축 후 MDS 적용

In [ ]:
# ─── MDS 3D 투영 ───────────────────────────
print(f"[MDS] 3D 투영 중... (서브샘플 {N_MDS}개, 시간 소요 예상)")

# MDS용 균등 서브샘플링
rng_mds = np.random.default_rng(RANDOM_SEED)
per_class_mds = N_MDS // 10
idx_mds = []
for c in range(10):
    ci = np.where(y == c)[0]
    idx_mds.append(rng_mds.choice(ci, size=min(per_class_mds, len(ci)), replace=False))
idx_mds = np.concatenate(idx_mds)

x_mds_sub = x[idx_mds]
y_mds = y[idx_mds]

# 사전 PCA 압축 (784 → 50차원) — MDS 속도 개선
pca_pre = PCA(n_components=50, random_state=RANDOM_SEED)
x_pre = pca_pre.fit_transform(x_mds_sub)
print(f"  PCA 사전 압축: 784 → 50차원 완료")

t0 = time.time()
mds = MDS(
    n_components=3,
    metric=True,
    n_init=1,
    max_iter=300,
    random_state=RANDOM_SEED,
    normalized_stress="auto",
    n_jobs=-1,
)
mds_coords = mds.fit_transform(x_pre)
mds_elapsed = time.time() - t0

print(f"  완료 ({mds_elapsed:.1f}s)  |  Stress: {mds.stress_:.4f}")
print(f"  사용 샘플: {len(idx_mds)}개")

## 6. 인터랙티브 3D 비교 시각화

세 subplot을 **독립적으로 마우스 드래그(회전) / 스크롤(줌) / 호버(값 확인)** 할 수 있습니다.

In [ ]:
# ─── 3D subplot 트레이스 추가 헬퍼 ─────────
def add_scatter3d(fig, coords, labels, row, col, show_legend):
    for digit in range(10):
        mask = labels == digit
        fig.add_trace(
            go.Scatter3d(
                x=coords[mask, 0],
                y=coords[mask, 1],
                z=coords[mask, 2],
                mode="markers",
                name=f"숫자 {digit}",
                legendgroup=f"digit_{digit}",
                showlegend=show_legend,
                marker=dict(
                    size=2.5,
                    color=COLORS[digit],
                    opacity=0.75,
                    line=dict(width=0),
                ),
                hovertemplate=(
                    f"<b>클래스 {digit}</b><br>"
                    "x: %{x:.3f}<br>"
                    "y: %{y:.3f}<br>"
                    "z: %{z:.3f}"
                    "<extra></extra>"
                ),
            ),
            row=row, col=col,
        )


# ─── Figure 구성 ───────────────────────────
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        f"PCA  (설명분산 {pca_var.sum()*100:.1f}%,  {pca_elapsed:.1f}s)",
        f"LDA  (설명분산 {lda_var.sum()*100:.1f}%,  {lda_elapsed:.1f}s)",
        f"MDS  (Stress={mds.stress_:.3f},  n={len(idx_mds)},  {mds_elapsed:.1f}s)",
    ],
    horizontal_spacing=0.02,
)

add_scatter3d(fig, pca_coords, y,      row=1, col=1, show_legend=True)
add_scatter3d(fig, lda_coords, y,      row=1, col=2, show_legend=False)
add_scatter3d(fig, mds_coords, y_mds,  row=1, col=3, show_legend=False)

# ─── 공통 축 스타일 ────────────────────────
axis_style = dict(backgroundcolor="#F0F2F5", gridcolor="white", showticklabels=False)
scene_defaults = dict(
    xaxis=dict(title="", **axis_style),
    yaxis=dict(title="", **axis_style),
    zaxis=dict(title="", **axis_style),
    bgcolor="#F0F2F5",
    camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
)

fig.update_layout(
    title=dict(
        text=(
            "MNIST 차원축소 비교: PCA  vs  LDA  vs  MDS  (3D Interactive)<br>"
            "<sup>마우스 드래그: 회전 · 스크롤: 줌 · 호버: 값 확인 · 범례 클릭: 클래스 토글</sup>"
        ),
        x=0.5,
        font=dict(size=18),
    ),
    scene=scene_defaults,
    scene2=scene_defaults,
    scene3=scene_defaults,
    legend=dict(
        title=dict(text="숫자 클래스", font=dict(size=13)),
        itemsizing="constant",
        font=dict(size=12),
        x=1.01, y=0.5,
    ),
    width=1600,
    height=700,
    margin=dict(l=0, r=120, t=100, b=0),
    paper_bgcolor="white",
)

fig.show()

fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn")
print(f"✅ {OUTPUT_HTML} 저장 완료")

## 7. 기법별 특성 요약

| | PCA | LDA | MDS |
|---|---|---|---|
| **지도 여부** | 비지도 | 지도 | 비지도 |
| **선형 여부** | 선형 | 선형 | 비선형 가능 |
| **보존 대상** | 분산 | 클래스 분리도 | 거리 구조 |
| **계산 복잡도** | O(n·d²) | O(n·d²) | **O(n²·d)** |
| **클래스 분리** | 보통 | **우수** | 보통 |
| **대규모 데이터** | 적합 | 적합 | **부적합** |

> **결론:** 레이블이 있다면 LDA가 가장 선명한 클래스 분리를 보여줍니다.  
> 비지도 상황에서는 PCA가 속도와 품질 면에서 가장 실용적입니다.